In [ ]:
from utils import get_stage_packages
get_stage_packages()
! pip install -r pip-requirements.txt

Processing ./dist/january_ml-0.0.1-py3-none-any.whl (from -r project-requirements.txt (line 1))
january-ml is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [ ]:
from january_ml.utils import load_config, version_data
from utils import get_data

from sklearn.linear_model import LinearRegression

from snowflake.snowpark.session import Session
from snowflake.ml.registry import Registry
from snowflake.ml.dataset import Dataset

config = load_config('example_project')

/opt/anaconda3/envs/jan/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create session
session = Session.builder.getOrCreate()

In [ ]:
# Get Data
df = get_data(config["N_SAMPLES_TRAIN"], config["N_FEATURES"])
sdf = session.create_dataframe(df)
sdf.show()

--------------------------------------------------------------------------------------------------------------------------------------
|"x_0"                |"x_1"                |"x_2"                |"x_3"                 |"x_4"                |"y"                  |
--------------------------------------------------------------------------------------------------------------------------------------
|0.7739560485559633   |0.4388784397520523   |0.8585979199113825   |0.6973680290593639    |0.09417734788764953  |0.9756223516367559   |
|0.761139701990353    |0.7860643052769538   |0.12811363267554587  |0.45038593789556713   |0.37079802423258124  |0.9267649888486018   |
|0.6438651200806645   |0.82276161327083     |0.44341419882733113  |0.2272387217847769    |0.5545847870158348   |0.06381725610417532  |
|0.8276311719925821   |0.6316643991220648   |0.7580877400853738   |0.35452596812986836   |0.9706980243949033   |0.8931211213221977   |
|0.7783834970737619   |0.19463870785196757  |0.46672100

In [ ]:
# Save as dataset
ds_name = config["TRAIN_DATA_NAME"]
ds = Dataset.create(session=session, name=ds_name, exist_ok=True)
version_name = version_data(sdf)
ds_version = ds.create_version(version=version_name, input_dataframe=sdf, label_cols=["y"])

In [ ]:
# Train Model

X = df.drop(columns=["y"])
y = df[["y"]]

lr = LinearRegression()
lr.fit(X, y)

In [ ]:
# Register model
reg = Registry(session=session)

model_name = config["MODEL_NAME"]
mv = reg.log_model(
    model=lr, 
    model_name=model_name, 
    sample_input_data=X, 
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"] )

reg.show_models()


In [ ]:
# promote to default
base_model = reg.get_model(model_name)
base_model.default = mv